# Changing-Look AGN Precursor Processing

Draft notebook-native workflow for the CLAGN precursor search. All draft code specific to this project lives in this notebook for now: source definitions, catalog construction, photometry proxy matching, ASAS-SN source resolution, light-curve fetching, cleaning, and first-pass metrics.


## Reusable Nuclear Context Layer

This notebook is now backed by the reusable `malca.nuclear` context and scoring API. Keep exploratory plots and catalog checks here, but put shared AGN/TDE/CLAGN enrichment, scoring, and review-export behavior in modules.


In [ ]:
from malca.nuclear import NuclearContextConfig, normalize_nuclear_targets, run_nuclear_context, score_nuclear_candidates
from malca.nuclear.features import compute_lightcurve_feature_table, compute_nuclear_lightcurve_features

# Example:
# config = NuclearContextConfig(run_dir=RUN_DIR)
# nuclear_context = run_nuclear_context(targets, config)


In [1]:
from __future__ import annotations

import io
import math
import re
import signal
import tarfile
import time
import urllib.request
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Callable

import astropy.units as u
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from astropy.coordinates import SkyCoord
from astropy.stats import bayesian_blocks
from astroquery.ipac.ned import Ned
from astroquery.simbad import Simbad
from astroquery.vizier import Vizier
from IPython.display import Markdown, display


def find_repo_root(start: Path | None = None) -> Path:
    here = (start or Path.cwd()).resolve()
    for candidate in (here, *here.parents):
        if (candidate / 'pyproject.toml').exists() and (candidate / 'malca').exists():
            return candidate
    return Path.cwd().resolve()

REPO_ROOT = find_repo_root()
REPO_ROOT


PosixPath('/Users/calder/code/malca')

In [ ]:
# Configuration. Keep network-backed work gated while this is in draft form.
INPUT_DIR = REPO_ROOT / 'input' / 'clagn'
CACHE_DIR = INPUT_DIR / 'cache'
OUTPUT_DIR = REPO_ROOT / 'output' / 'runs' / 'clagn_precursor'
LIGHTCURVE_DIR = OUTPUT_DIR / 'lightcurves'
PROCESSED_DIR = OUTPUT_DIR / 'processed'
CLEANED_LIGHTCURVE_DIR = PROCESSED_DIR / 'cleaned_lightcurves'

SOURCE_REGISTRY_PATH = INPUT_DIR / 'source_registry.csv'
ALL_CATALOG_PATH = INPUT_DIR / 'clagn_literature_sources.csv'
SENSITIVE_CATALOG_PATH = INPUT_DIR / 'clagn_asassn_sensitive_proxy.csv'
FAILURES_PATH = INPUT_DIR / 'clagn_catalog_failures.csv'
LYU22_SOURCE_TABLE = CACHE_DIR / 'lyu22_source_table.tex'

ASASSN_G_LIMIT = 18.5
ASASSN_V_LIMIT = 17.0
DEDUP_RADIUS_ARCSEC = 3.0
PHOTOMETRY_RADIUS_ARCSEC = 2.0
SKYPATROL_RADIUS_ARCSEC = 5.0
SKYPATROL_BACKEND = 'skypatrol1'
ARCHIVE_TIMEOUT_SECONDS = 45
MAX_OBJECTS = None

RUN_CATALOG_BUILD = False
RUN_PHOTOMETRY_PROXY_MATCH = False
RUN_ASASSN_RESOLUTION = False
RUN_ASASSN_FETCH = False
RUN_LIGHTCURVE_PROCESSING = False
RUN_TRANSITION_METADATA = False
RUN_PRECURSOR_SCIENCE = False
RUN_REVIEW_PLOTS = False

ROLLING_ANOMALY_WINDOW = 25
ROLLING_ANOMALY_SIGMA = 5.0
MIN_SEASON_POINTS = 5
PRECURSOR_WINDOWS_YEARS = (1, 2, 5)
REVIEW_TOP_N = 25

for path in (INPUT_DIR, CACHE_DIR, OUTPUT_DIR, LIGHTCURVE_DIR, PROCESSED_DIR, CLEANED_LIGHTCURVE_DIR):
    path.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_colwidth', 120)


## Source Registry

`manual_pending` entries are listed here so the missing 2023-2025 work is visible in the notebook, but they are not automatically ingested yet. Add table-specific parsers for those before calling the parent list complete.

In [3]:
SOURCE_REGISTRY = [
    dict(key='lamassa15', label='LaMassa 2015', bibcode='2015ApJ...800..144L', year=2015, status='confirmed', getter='ned_first_row', notes='First changing-look quasar SDSS J015957.64+003310.5.'),
    dict(key='macleod16', label='MacLeod 2016', bibcode='2016MNRAS.457..389M', year=2016, status='confirmed', getter='static_table', notes='SDSS repeat-spectroscopy changing-look quasars.'),
    dict(key='ruan16', label='Ruan 2016', bibcode='2016ApJ...826..188R', year=2016, status='confirmed', getter='ned_refcode', notes='Three changing-look quasars.'),
    dict(key='yang18', label='Yang 2018', bibcode='2018ApJ...862..109Y', year=2018, status='confirmed', getter='ned_refcode', notes='New CLAGN plus literature objects in NED reference list.'),
    dict(key='macleod19', label='MacLeod 2019', bibcode='2019ApJ...874....8M', year=2019, status='confirmed', getter='vizier_filtered', notes='CLQ/Nsigma-filtered VizieR table.'),
    dict(key='sheng20', label='Sheng 2020', bibcode='2020ApJ...889...46S', year=2020, status='confirmed', getter='ned_selected_rows', notes='Three changing-look quasars selected from NED reference result.'),
    dict(key='lyu22', label='Lyu 2022', bibcode='2022ApJ...927..227L', year=2022, status='confirmed', getter='arxiv_source_table', notes='Known CLAGN sample assembled for IR echoes.'),
    dict(key='green22', label='Green 2022', bibcode='2022ApJ...933..180G', year=2022, status='confirmed', getter='vizier_filtered', notes='SDSS-V pilot/CLAGN sample.'),
    dict(key='lopeznavas22', label='Lopez-Navas 2022', bibcode='2022MNRAS.513L..57L', year=2022, status='confirmed', getter='static_table', notes='Confirmed new CLAGN from ZTF/ALeRCE candidate list.'),
    dict(key='hon22', label='Hon 2022', bibcode='2022MNRAS.511...54H', year=2022, status='confirmed', getter='static_table', notes='SkyMapper colours II CLAGN; names encode coordinates.'),
    dict(key='wang22', label='Wang 2022', bibcode='2022RAA....22k5014W', year=2022, status='confirmed', getter='manual_pending', notes='Add source-table parser.'),
    dict(key='temple23_bass', label='Temple 2023', bibcode='2022MNRAS.518.2938T', year=2023, status='confirmed', getter='manual_pending', notes='BASS XXXIX Swift-BAT changing-look spectra.'),
    dict(key='yang23_desi_edr', label='Yang 2023', bibcode='2023arXiv230708289Y', year=2023, status='confirmed', getter='manual_pending', notes='DESI early-data CLAGN sample.'),
    dict(key='zeltyn24', label='Zeltyn 2024', bibcode='2024ApJ...966...85Z', year=2024, status='confirmed', getter='manual_pending', notes='First-year SDSS-V CLAGN sample.'),
    dict(key='yang24_desi_dr1', label='Yang 2024', bibcode='2024arXiv240800402Y', year=2024, status='confirmed', getter='manual_pending', notes='DESI DR1/SDSS DR16 CLAGN catalog.'),
    dict(key='wolf24_6df_atlas', label='Wolf 2024', bibcode='2024MNRAS.535.2322W', year=2024, status='confirmed', getter='manual_pending', notes='6dF/ATLAS z<0.1 CLAGN search.'),
    dict(key='li25_turnon', label='Li 2025', bibcode='2025ApJ...980...91L', year=2025, status='confirmed', getter='manual_pending', notes='Large turn-on CLAGN sample from SDSS galaxies.'),
]
source_registry = pd.DataFrame(SOURCE_REGISTRY)
source_registry.to_csv(SOURCE_REGISTRY_PATH, index=False)
source_registry


,key,label,bibcode,year,status,getter,notes
0,lamassa15,LaMassa 2015,2015ApJ...800..144L,2015,confirmed,ned_first_row,First changing-look quasar SDSS J015957.64+003310.5.
1,macleod16,MacLeod 2016,2016MNRAS.457..389M,2016,confirmed,static_table,SDSS repeat-spectroscopy changing-look quasars.
2,ruan16,Ruan 2016,2016ApJ...826..188R,2016,confirmed,ned_refcode,Three changing-look quasars.
3,yang18,Yang 2018,2018ApJ...862..109Y,2018,confirmed,ned_refcode,New CLAGN plus literature objects in NED reference list.
4,macleod19,MacLeod 2019,2019ApJ...874....8M,2019,confirmed,vizier_filtered,CLQ/Nsigma-filtered VizieR table.
5,sheng20,Sheng 2020,2020ApJ...889...46S,2020,confirmed,ned_selected_rows,Three changing-look quasars selected from NED reference result.
6,lyu22,Lyu 2022,2022ApJ...927..227L,2022,confirmed,arxiv_source_table,Known CLAGN sample assembled for IR echoes.
7,green22,Green 2022,2022ApJ...933..180G,2022,confirmed,vizier_filtered,SDSS-V pilot/CLAGN sample.
8,lopeznavas22,Lopez-Navas 2022,2022MNRAS.513L..57L,2022,confirmed,static_table,Confirmed new CLAGN from ZTF/ALeRCE candidate list.
9,hon22,Hon 2022,2022MNRAS.511...54H,2022,confirmed,static_table,SkyMapper colours II CLAGN; names encode coordinates.


## Static Paper Tables And Catalog Parsers

These helpers are intentionally inline. They can move into package modules only after the draft stabilizes.

In [ ]:
MACLEOD16_OBJECTS = [
    ('SDSS J002311.06+003517.5', 0.422), ('SDSS J015957.64+003310.4', 0.312),
    ('SDSS J022556.07+003026.7', 0.504), ('SDSS J022652.24-003916.5', 0.625),
    ('SDSS J100220.17+450927.3', 0.400), ('SDSS J102152.34+464515.6', 0.204),
    ('SDSS J132457.29+480241.2', 0.272), ('SDSS J214613.31+000930.8', 0.621),
    ('SDSS J225240.37+010958.7', 0.534), ('SDSS J233317.38-002303.4', 0.513),
]

LOPEZNAVAS22_CONFIRMED = [
    ('ZTF19abixawb', 'SDSS J001014.86+000820.7', 0.1022),
    ('ZTF20abshfkf', 'SDSS J011311.82+013542.4', 0.2375),
    ('ZTF18accdhxv', 'SDSS J075544.35+192336.3', 0.1083),
    ('ZTF19aalxuyo', 'SDSS J081240.76+071528.5', 0.0849),
]

HON22_OBJECTS = [
    ('J0041321-223838', 0.06307, 'turn_on'), ('J0130213-460145', 0.11152, 'turn_on'),
    ('J0345125-393429', 0.04320, 'turn_on'), ('J0454341-200507', 0.07527, 'turn_on'),
    ('J0613243-290023', 0.07051, 'turn_on'), ('J0917272-645628', 0.08600, 'turn_on_multi'),
    ('J1008486-095451', 0.05725, 'turn_on'), ('J1109146-125554', 0.02551, 'turn_on_atypical'),
    ('J1824525-432858', 0.07186, 'turn_on'), ('J2007513-110835', 0.03109, 'turn_on'),
    ('J2037599-502334', 0.06311, 'turn_on'), ('J2309192-322957', 0.05417, 'turn_on'),
    ('J2330323-022745', 0.03322, 'turn_on'), ('J1340153-045332', 0.08654, 'turn_off_multi'),
    ('J2046446-012208', 0.02519, 'turn_off'), ('J0010100-044238', 0.02937, 'serendipitous'),
    ('J0014181-051904', 0.08398, 'serendipitous'), ('J0248345-720831', 0.07572, 'serendipitous'),
    ('J0458403-215931', 0.03944, 'serendipitous'), ('J0519358-323928', 0.01258, 'serendipitous'),
    ('J0612386-354307', 0.04415, 'serendipitous'), ('J1538448-032248', 0.02374, 'serendipitous'),
    ('J1721342-544315', 0.06234, 'serendipitous'), ('J1839358-354124', 0.04567, 'serendipitous'),
    ('J1949137-184716', 0.08113, 'serendipitous'), ('J1958565-422230', 0.03203, 'serendipitous'),
    ('J1406507-244250', 0.04570, 'clnls1'),
]

@dataclass(frozen=True)
class SourceSpec:
    key: str
    label: str
    bibcode: str
    getter: str
    rows: tuple[int, ...] | None = None
    optional: bool = False

SOURCE_SPECS = [
    SourceSpec('lamassa15', 'LaMassa 2015', '2015ApJ...800..144L', 'ned', rows=(0,)),
    SourceSpec('macleod16', 'MacLeod 2016', '2016MNRAS.457..389M', 'macleod16_static'),
    SourceSpec('ruan16', 'Ruan 2016', '2016ApJ...826..188R', 'ned'),
    SourceSpec('yang18', 'Yang 2018', '2018ApJ...862..109Y', 'ned'),
    SourceSpec('macleod19', 'MacLeod 2019', '2019ApJ...874....8M', 'macleod19_vizier'),
    SourceSpec('sheng20', 'Sheng 2020', '2020ApJ...889...46S', 'ned', rows=(0, 1, 3)),
    SourceSpec('lyu22', 'Lyu 2022', '2022ApJ...927..227L', 'lyu22_arxiv'),
    SourceSpec('green22', 'Green 2022', '2022ApJ...933..180G', 'green22_vizier'),
    SourceSpec('lopeznavas22', 'Lopez-Navas 2022', '2022MNRAS.513L..57L', 'lopeznavas22_static'),
    SourceSpec('hon22', 'Hon 2022', '2022MNRAS.511...54H', 'hon22_static'),
    SourceSpec('wang22', 'Wang 2022', '2022RAA....22k5014W', 'ned', optional=True),
]

class NotebookTimeoutError(RuntimeError):
    pass

def _timeout_handler(_signum, _frame):
    raise NotebookTimeoutError('archive query timed out')

def with_timeout(timeout_s: int, func: Callable, *args, **kwargs):
    old_handler = signal.signal(signal.SIGALRM, _timeout_handler)
    signal.alarm(int(timeout_s))
    try:
        return func(*args, **kwargs)
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old_handler)

def finite_float(value) -> float:
    try:
        x = float(value)
    except Exception:
        return math.nan
    return x if math.isfinite(x) else math.nan

def add_catalog_row(rows, *, name, ra_deg, dec_deg, source_key, source_label, bibcode, redshift=math.nan, clagn_type=''):
    ra, dec = finite_float(ra_deg), finite_float(dec_deg)
    if math.isfinite(ra) and math.isfinite(dec):
        rows.append(dict(name=str(name).strip(), ra_deg=ra, dec_deg=dec, redshift=finite_float(redshift), source_key=source_key, source_label=source_label, bibcode=bibcode, clagn_type=str(clagn_type).strip()))

def parse_sdss_j_coord(value) -> SkyCoord:
    text = str(value).strip()
    if text.startswith('SDSS'):
        text = text.split()[-1]
    if text.startswith('J'):
        text = text[1:]
    text = text.replace(' ', '')
    sign_pos = max(text.find('+'), text.find('-'))
    if sign_pos <= 0:
        raise ValueError(f'Cannot parse coordinate from {value!r}')
    ra, dec = text[:sign_pos], text[sign_pos:]
    return SkyCoord(f'{ra[0:2]} {ra[2:4]} {ra[4:]} {dec[0:3]} {dec[3:5]} {dec[5:]}', unit=(u.hourangle, u.deg), frame='icrs')

def parse_embedded_j_coord(value) -> SkyCoord:
    text = str(value).strip()
    match = re.search(r'J(\d{6,8}(?:\.\d+)?)([+-]\d{6,7}(?:\.\d+)?)', text)
    if not match:
        return parse_sdss_j_coord(text)
    ra, dec = match.group(1), match.group(2)
    ra_sec = ra[4:] if '.' in ra else (f'{ra[4:6]}.{ra[6:]}' if len(ra) > 6 else ra[4:6])
    dec_sign, dec_digits = dec[0], dec[1:]
    dec_sec = dec_digits[4:] if '.' in dec_digits else (f'{dec_digits[4:6]}.{dec_digits[6:]}' if len(dec_digits) > 6 else dec_digits[4:6])
    return SkyCoord(f'{ra[0:2]} {ra[2:4]} {ra_sec} {dec_sign}{dec_digits[0:2]} {dec_digits[2:4]} {dec_sec}', unit=(u.hourangle, u.deg), frame='icrs')



In [ ]:
def rows_from_ned(spec: SourceSpec, *, timeout_s: int) -> list[dict]:
    table = with_timeout(timeout_s, Ned.query_refcode, spec.bibcode)
    if spec.rows is not None:
        table = table[list(spec.rows)]
    rows = []
    for row in table:
        add_catalog_row(rows, name=row['Object Name'], ra_deg=row['RA'], dec_deg=row['DEC'], redshift=row['Redshift'] if 'Redshift' in row.colnames else math.nan, source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode)
    return rows

def rows_from_macleod16_static(spec: SourceSpec, *, timeout_s: int) -> list[dict]:
    rows = []
    for name, redshift in MACLEOD16_OBJECTS:
        coord = parse_sdss_j_coord(name)
        add_catalog_row(rows, name=name, ra_deg=coord.ra.deg, dec_deg=coord.dec.deg, redshift=redshift, source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode)
    return rows

def rows_from_macleod19_vizier(spec: SourceSpec, *, timeout_s: int) -> list[dict]:
    def query():
        Vizier.ROW_LIMIT = -1
        catalogs = Vizier.get_catalogs(Vizier.find_catalogs(spec.bibcode).keys())
        return catalogs[0]
    table = with_timeout(timeout_s, query)
    clq_col = 'CLQ_' if 'CLQ_' in table.colnames else 'CLQ?'
    selected = table[(np.asarray(table[clq_col]) > 0) & (np.asarray(table['Nsigma']) > 3)]
    rows = []
    for row in selected:
        add_catalog_row(rows, name=row['SDSS'] if 'SDSS' in row.colnames else f"{row['_RA']:.6f},{row['_DE']:.6f}", ra_deg=row['_RA'], dec_deg=row['_DE'], redshift=row['z'] if 'z' in row.colnames else math.nan, source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode)
    return rows

def rows_from_green22_vizier(spec: SourceSpec, *, timeout_s: int) -> list[dict]:
    def query():
        Vizier.ROW_LIMIT = -1
        return Vizier.get_catalogs(['J/ApJ/933/180'])[0].to_pandas()
    frame = with_timeout(timeout_s, query)
    frame = frame[frame['Notes'].astype(str).str.contains('CLQ', na=False)].copy()
    rows = []
    for _, row in frame.iterrows():
        coord = parse_sdss_j_coord(row['SDSS'])
        add_catalog_row(rows, name=row['SDSS'], ra_deg=coord.ra.deg, dec_deg=coord.dec.deg, source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode)
    return rows

def _clean_tex_value(value: str) -> str:
    value = re.sub(r'\$.*', '', value.strip()).strip()
    return value.replace(r'\pm', '+/-').strip()

def _download_lyu22_source_table(timeout_s: int) -> str:
    if LYU22_SOURCE_TABLE.exists():
        return LYU22_SOURCE_TABLE.read_text()
    def download() -> bytes:
        request = urllib.request.Request('https://arxiv.org/src/2202.02718', headers={'User-Agent': 'malca-clagn-notebook/0.1'})
        with urllib.request.urlopen(request, timeout=timeout_s) as response:
            return response.read()
    payload = with_timeout(timeout_s, download)
    with tarfile.open(fileobj=io.BytesIO(payload), mode='r:*') as tar:
        member = next((m for m in tar.getmembers() if m.name.endswith('source_table.tex')), None)
        if member is None:
            raise FileNotFoundError('source_table.tex not found in Lyu 2022 source')
        handle = tar.extractfile(member)
        text = handle.read().decode('utf-8')
    LYU22_SOURCE_TABLE.write_text(text)
    return text

def rows_from_lyu22_arxiv(spec: SourceSpec, *, timeout_s: int) -> list[dict]:
    parsed, in_data = [], False
    for line in _download_lyu22_source_table(timeout_s).splitlines():
        line = line.strip()
        if line == r'\startdata':
            in_data = True
            continue
        if line == r'\enddata':
            break
        if in_data and '&' in line:
            parts = [_clean_tex_value(part) for part in line.rstrip('\\').split('&')]
            if len(parts) >= 3:
                parsed.append(dict(name=parts[0], redshift=finite_float(parts[1]), clagn_type=parts[2]))
    coord_by_name, unresolved = {}, []
    for row in parsed:
        try:
            coord_by_name[row['name']] = parse_embedded_j_coord(row['name'])
        except Exception:
            unresolved.append(row['name'])
    if unresolved:
        try:
            simbad_rows = with_timeout(timeout_s, Simbad.query_objects, unresolved)
            if simbad_rows is not None:
                for simbad_row in simbad_rows:
                    coord_by_name[str(simbad_row['user_specified_id']).strip()] = SkyCoord(simbad_row['ra'], simbad_row['dec'], unit='deg', frame='icrs')
        except Exception as exc:
            warnings.warn(f'Lyu 2022 SIMBAD coordinate batch failed: {exc}')
    rows = []
    for row in parsed:
        coord = coord_by_name.get(row['name'])
        if coord is not None:
            add_catalog_row(rows, name=row['name'], ra_deg=coord.ra.deg, dec_deg=coord.dec.deg, redshift=row['redshift'], source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode, clagn_type=row['clagn_type'])
    return rows

def rows_from_lopeznavas22_static(spec: SourceSpec, *, timeout_s: int) -> list[dict]:
    rows = []
    for ztf_id, sdss_name, redshift in LOPEZNAVAS22_CONFIRMED:
        coord = parse_sdss_j_coord(sdss_name)
        add_catalog_row(rows, name=f'{ztf_id} / {sdss_name}', ra_deg=coord.ra.deg, dec_deg=coord.dec.deg, redshift=redshift, source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode)
    return rows

def rows_from_hon22_static(spec: SourceSpec, *, timeout_s: int) -> list[dict]:
    rows = []
    for name, redshift, clagn_type in HON22_OBJECTS:
        coord = parse_embedded_j_coord(name)
        add_catalog_row(rows, name=name, ra_deg=coord.ra.deg, dec_deg=coord.dec.deg, redshift=redshift, source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode, clagn_type=clagn_type)
    return rows

GETTERS = {
    'ned': rows_from_ned,
    'macleod16_static': rows_from_macleod16_static,
    'macleod19_vizier': rows_from_macleod19_vizier,
    'green22_vizier': rows_from_green22_vizier,
    'lyu22_arxiv': rows_from_lyu22_arxiv,
    'lopeznavas22_static': rows_from_lopeznavas22_static,
    'hon22_static': rows_from_hon22_static,
}


In [6]:
def collect_literature_rows(*, timeout_s: int, include_optional: bool = False) -> tuple[pd.DataFrame, pd.DataFrame]:
    all_rows, failures = [], []
    for spec in SOURCE_SPECS:
        if spec.optional and not include_optional:
            continue
        print(f'[source] {spec.key}: {spec.label}', flush=True)
        start = time.time()
        try:
            rows = GETTERS[spec.getter](spec, timeout_s=timeout_s)
            print(f'  rows={len(rows)} elapsed={time.time() - start:.1f}s', flush=True)
            all_rows.extend(rows)
        except Exception as exc:
            failures.append(dict(source_key=spec.key, source_label=spec.label, bibcode=spec.bibcode, error=f'{type(exc).__name__}: {exc}'))
            print(f'  FAILED: {type(exc).__name__}: {exc}', flush=True)
    return pd.DataFrame(all_rows), pd.DataFrame(failures)

def deduplicate_sources(frame: pd.DataFrame, *, radius_arcsec: float = DEDUP_RADIUS_ARCSEC) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    coords = SkyCoord(frame['ra_deg'].to_numpy() * u.deg, frame['dec_deg'].to_numpy() * u.deg)
    assigned = np.full(len(frame), -1, dtype=int)
    groups = []
    for idx in np.argsort(frame['ra_deg'].to_numpy()):
        if assigned[idx] >= 0:
            continue
        members = np.where((assigned < 0) & (coords[idx].separation(coords).arcsec <= radius_arcsec))[0].tolist()
        for member in members:
            assigned[member] = len(groups)
        groups.append(members)
    rows = []
    for group_id, members in enumerate(groups, start=1):
        subset = frame.iloc[members].copy()
        names = sorted({str(x) for x in subset['name'].dropna() if str(x).strip()})
        redshifts = [finite_float(x) for x in subset['redshift']]
        redshifts = [x for x in redshifts if math.isfinite(x)]
        rows.append(dict(
            clagn_id=f'CLAGN-{group_id:04d}', name=names[0] if names else str(subset.iloc[0]['name']), aliases='|'.join(names),
            ra_deg=float(np.nanmedian(subset['ra_deg'].to_numpy(dtype=float))), dec_deg=float(np.nanmedian(subset['dec_deg'].to_numpy(dtype=float))),
            redshift=float(np.nanmedian(redshifts)) if redshifts else math.nan, n_source_mentions=int(len(subset)),
            source_keys='|'.join(sorted(set(subset['source_key'].astype(str)))), source_labels='|'.join(sorted(set(subset['source_label'].astype(str)))),
            bibcodes='|'.join(sorted(set(subset['bibcode'].astype(str)))), clagn_types='|'.join(sorted({str(x) for x in subset['clagn_type'].dropna() if str(x).strip()})),
        ))
    out = pd.DataFrame(rows).sort_values(['ra_deg', 'dec_deg']).reset_index(drop=True)
    out['clagn_id'] = [f'CLAGN-{i:04d}' for i in range(1, len(out) + 1)]
    return out

PHOTOMETRY_COLUMNS = ['ps1_gmag','ps1_rmag','ps1_imag','ps1_sep_arcsec','smss_gpsf','smss_gpetro','smss_rpsf','smss_rpetro','smss_sep_arcsec','gaia_gmag','gaia_bpmag','gaia_rpmag','gaia_sep_arcsec','g_proxy_mag','g_proxy_source','asassn_g_sensitive_proxy','asassn_g_limit','asassn_v_limit']

def nearest_table_row(tables, coord: SkyCoord, ra_col: str, dec_col: str):
    if len(tables) == 0 or len(tables[0]) == 0:
        return None, math.nan
    table = tables[0]
    sep = coord.separation(SkyCoord(table[ra_col], table[dec_col], unit='deg')).arcsec
    idx = int(np.argmin(sep))
    return table[idx], float(sep[idx])

def query_photometry_for_coord(coord: SkyCoord, *, timeout_s: int = ARCHIVE_TIMEOUT_SECONDS) -> dict:
    out = {col: math.nan for col in PHOTOMETRY_COLUMNS}
    out.update(g_proxy_source='', asassn_g_sensitive_proxy=False, asassn_g_limit=ASASSN_G_LIMIT, asassn_v_limit=ASASSN_V_LIMIT)
    query_specs = [
        ('ps1', 'II/349/ps1', ['RAJ2000','DEJ2000','gmag','rmag','imag'], 'RAJ2000', 'DEJ2000'),
        ('smss', 'II/379/smssdr4', ['RAICRS','DEICRS','gPSF','gPetro','rPSF','rPetro'], 'RAICRS', 'DEICRS'),
        ('gaia', 'I/355/gaiadr3', ['RA_ICRS','DE_ICRS','Gmag','BPmag','RPmag'], 'RA_ICRS', 'DE_ICRS'),
    ]
    for key, catalog, columns, ra_col, dec_col in query_specs:
        try:
            def query():
                return Vizier(columns=columns, row_limit=5).query_region(coord, radius=PHOTOMETRY_RADIUS_ARCSEC * u.arcsec, catalog=catalog)
            row, sep = nearest_table_row(with_timeout(timeout_s, query), coord, ra_col, dec_col)
        except Exception:
            continue
        if row is None:
            continue
        if key == 'ps1':
            out.update(ps1_gmag=finite_float(row['gmag']), ps1_rmag=finite_float(row['rmag']), ps1_imag=finite_float(row['imag']), ps1_sep_arcsec=sep)
        elif key == 'smss':
            out.update(smss_gpsf=finite_float(row['gPSF']), smss_gpetro=finite_float(row['gPetro']), smss_rpsf=finite_float(row['rPSF']), smss_rpetro=finite_float(row['rPetro']), smss_sep_arcsec=sep)
        elif key == 'gaia':
            out.update(gaia_gmag=finite_float(row['Gmag']), gaia_bpmag=finite_float(row['BPmag']), gaia_rpmag=finite_float(row['RPmag']), gaia_sep_arcsec=sep)
    for source in ('ps1_gmag', 'smss_gpetro', 'smss_gpsf', 'gaia_gmag'):
        value = finite_float(out[source])
        if math.isfinite(value):
            out['g_proxy_mag'], out['g_proxy_source'] = value, source
            break
    out['asassn_g_sensitive_proxy'] = bool(math.isfinite(finite_float(out['g_proxy_mag'])) and finite_float(out['g_proxy_mag']) <= ASASSN_G_LIMIT)
    return out

def add_photometry_proxy(frame: pd.DataFrame, *, timeout_s: int = ARCHIVE_TIMEOUT_SECONDS) -> pd.DataFrame:
    base = frame.drop(columns=[c for c in PHOTOMETRY_COLUMNS if c in frame.columns], errors='ignore').copy()
    rows = []
    for i, row in base.iterrows():
        print(f"[phot] {i + 1}/{len(base)} {row['clagn_id']} {row['name']}", flush=True)
        rows.append(query_photometry_for_coord(SkyCoord(row['ra_deg'] * u.deg, row['dec_deg'] * u.deg), timeout_s=timeout_s))
    return pd.concat([base.reset_index(drop=True), pd.DataFrame(rows).reset_index(drop=True)], axis=1)


In [7]:
# Build or load the CLAGN parent catalog.
if RUN_CATALOG_BUILD:
    raw_catalog, source_failures = collect_literature_rows(timeout_s=ARCHIVE_TIMEOUT_SECONDS)
    clagn_catalog = deduplicate_sources(raw_catalog)
    source_failures.to_csv(FAILURES_PATH, index=False)
    clagn_catalog.to_csv(ALL_CATALOG_PATH, index=False)
elif ALL_CATALOG_PATH.exists():
    clagn_catalog = pd.read_csv(ALL_CATALOG_PATH)
    source_failures = pd.read_csv(FAILURES_PATH) if FAILURES_PATH.exists() else pd.DataFrame()
else:
    clagn_catalog = pd.DataFrame()
    source_failures = pd.DataFrame()

if RUN_PHOTOMETRY_PROXY_MATCH and not clagn_catalog.empty:
    clagn_catalog = add_photometry_proxy(clagn_catalog, timeout_s=ARCHIVE_TIMEOUT_SECONDS)
    clagn_catalog.to_csv(ALL_CATALOG_PATH, index=False)
    clagn_catalog[clagn_catalog['asassn_g_sensitive_proxy'].astype(bool)].to_csv(SENSITIVE_CATALOG_PATH, index=False)

if SENSITIVE_CATALOG_PATH.exists():
    clagn_sensitive = pd.read_csv(SENSITIVE_CATALOG_PATH)
elif not clagn_catalog.empty and 'asassn_g_sensitive_proxy' in clagn_catalog.columns:
    clagn_sensitive = clagn_catalog[clagn_catalog['asassn_g_sensitive_proxy'].astype(bool)].copy()
else:
    clagn_sensitive = pd.DataFrame()

display(Markdown(f'**Catalog rows:** {len(clagn_catalog):,} total; **ASAS-SN proxy-sensitive rows:** {len(clagn_sensitive):,}'))
if not source_failures.empty:
    display(source_failures)
clagn_sensitive.head(10)


**Catalog rows:** 0 total; **ASAS-SN proxy-sensitive rows:** 0

""


## ASAS-SN Source Resolution And Light-Curve Processing

This stage resolves the proxy-sensitive CLAGN coordinates to a nearby Sky Patrol source, optionally downloads a light curve, cleans it, and computes first-pass global metrics. Transition-window metrics will come after the catalog has spectrum dates.

In [8]:
from malca.io.fetch import cone_search, download_lightcurve_by_id

RESOLUTION_COLUMNS = ['asassn_id','asassn_source_id','asassn_sep_arcsec','asassn_mean_vmag','asassn_epochs','lc_path','lc_fetch_status','lc_fetch_error']

def resolve_skypatrol_source(row: pd.Series, *, radius_arcsec: float = SKYPATROL_RADIUS_ARCSEC, backend: str = SKYPATROL_BACKEND) -> dict:
    out = {col: np.nan for col in RESOLUTION_COLUMNS}
    out.update(lc_fetch_status='unfetched', lc_fetch_error='')
    try:
        matches = cone_search(float(row['ra_deg']), float(row['dec_deg']), radius_arcsec=radius_arcsec, backend=backend)
    except Exception as exc:
        out.update(lc_fetch_status='resolve_failed', lc_fetch_error=f'{type(exc).__name__}: {exc}')
        return out
    if matches.empty:
        out['lc_fetch_status'] = 'no_match'
        return out
    sep_col = 'sp1_sep_arcsec' if 'sp1_sep_arcsec' in matches.columns else 'sep_arcsec'
    if sep_col in matches.columns:
        matches = matches.sort_values(sep_col)
    best = matches.iloc[0]
    source_id = str(best.get('asas_sn_id') or best.get('source_id') or '').strip()
    out.update(asassn_id=source_id, asassn_source_id=str(best.get('source_id') or source_id), asassn_sep_arcsec=finite_float(best.get(sep_col, np.nan)), asassn_mean_vmag=finite_float(best.get('mean_vmag', np.nan)), asassn_epochs=finite_float(best.get('epochs', np.nan)))
    return out

def resolve_catalog_sources(frame: pd.DataFrame, *, max_objects: int | None = MAX_OBJECTS) -> pd.DataFrame:
    work = frame.head(max_objects).copy() if max_objects else frame.copy()
    rows = []
    for i, row in work.iterrows():
        print(f"[resolve] {i + 1}/{len(work)} {row.get('clagn_id', '')} {row.get('name', '')}", flush=True)
        rows.append(resolve_skypatrol_source(row))
    return pd.concat([work.reset_index(drop=True), pd.DataFrame(rows).reset_index(drop=True)], axis=1)

def fetch_resolved_lightcurves(frame: pd.DataFrame, *, backend: str = SKYPATROL_BACKEND, refresh_cache: bool = False) -> pd.DataFrame:
    out = frame.copy()
    for col in ('lc_path', 'lc_fetch_status', 'lc_fetch_error'):
        if col not in out.columns:
            out[col] = ''
    for i, row in out.iterrows():
        asassn_id = str(row.get('asassn_id') or '').strip()
        if not asassn_id or asassn_id.lower() == 'nan':
            out.at[i, 'lc_fetch_status'] = 'no_asassn_id'
            continue
        print(f"[fetch] {i + 1}/{len(out)} {row.get('clagn_id', '')} {asassn_id}", flush=True)
        try:
            path, meta = download_lightcurve_by_id(asassn_id, cache_dir=LIGHTCURVE_DIR, backend=backend, refresh_cache=refresh_cache)
            out.at[i, 'lc_path'] = str(path)
            out.at[i, 'lc_fetch_status'] = 'ok'
            out.at[i, 'lc_fetch_error'] = ''
        except Exception as exc:
            out.at[i, 'lc_fetch_status'] = 'fetch_failed'
            out.at[i, 'lc_fetch_error'] = f'{type(exc).__name__}: {exc}'
    return out


In [9]:
def read_skypatrol_lightcurve(path: str | Path) -> pd.DataFrame:
    df = pd.read_csv(Path(path), comment='#')
    rename = {'JD':'jd','Flux':'flux','Flux Error':'flux_err','Mag':'mag','Mag Error':'mag_err','Limit':'limit','FWHM':'fwhm','Filter':'band','Quality':'quality','Camera':'camera'}
    df = df.rename(columns={k: v for k, v in rename.items() if k in df.columns}).copy()
    for col in ('jd','flux','flux_err','mag','mag_err','limit','fwhm'):
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    if 'band' not in df.columns:
        df['band'] = 'unknown'
    if 'quality' not in df.columns:
        df['quality'] = 'G'
    df['source_path'] = str(path)
    return df

def clean_asassn_lightcurve(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df.copy()
    out = df[pd.to_numeric(df['jd'], errors='coerce').notna()].copy()
    if 'quality' in out.columns:
        out = out[out['quality'].astype(str).str.upper().isin(['G', 'GOOD', '1', 'TRUE'])].copy()
    if 'flux' in out.columns:
        out = out[np.isfinite(out['flux'])].copy()
    if 'flux_err' in out.columns:
        out = out[(out['flux_err'].isna()) | (out['flux_err'] > 0)].copy()
    return out.sort_values('jd').reset_index(drop=True)

def robust_slope_per_year(x_jd: np.ndarray, y: np.ndarray) -> float:
    mask = np.isfinite(x_jd) & np.isfinite(y)
    if mask.sum() < 3:
        return math.nan
    x_year = (x_jd[mask] - np.nanmin(x_jd[mask])) / 365.25
    if np.nanmax(x_year) <= 0:
        return math.nan
    return float(np.polyfit(x_year, y[mask], 1)[0])

def safe_filename_token(value, *, fallback: str = 'row') -> str:
    text = str(value or '').strip()
    text = re.sub(r'[^A-Za-z0-9_.-]+', '_', text).strip('_')
    return text or fallback

def cleaned_lightcurve_path_for_row(row: pd.Series) -> Path:
    clagn_id = safe_filename_token(row.get('clagn_id', ''), fallback='clagn')
    asassn_id = safe_filename_token(row.get('asassn_id', ''), fallback='asassn')
    return CLEANED_LIGHTCURVE_DIR / f'{clagn_id}_{asassn_id}.clean.csv'

def write_cleaned_lightcurve(row: pd.Series, clean: pd.DataFrame) -> str:
    CLEANED_LIGHTCURVE_DIR.mkdir(parents=True, exist_ok=True)
    clean_path = cleaned_lightcurve_path_for_row(row)
    clean.to_csv(clean_path, index=False)
    return str(clean_path)

def summarize_clean_lightcurve(df: pd.DataFrame) -> dict:
    out = dict(lc_n_points=len(df), lc_n_bands=0, lc_jd_start=math.nan, lc_jd_end=math.nan, lc_span_days=math.nan, lc_median_mag=math.nan, lc_amp_mag_p95_p05=math.nan, lc_flux_median=math.nan, lc_flux_amp_p95_p05=math.nan, lc_mag_slope_per_year=math.nan, lc_flux_slope_per_year=math.nan, lc_bb_n_blocks=math.nan, lc_bb_largest_jump_flux=math.nan, clean_lc_path='')
    if df.empty:
        return out
    out.update(lc_n_bands=int(df['band'].nunique()) if 'band' in df.columns else 0, lc_jd_start=float(df['jd'].min()), lc_jd_end=float(df['jd'].max()), lc_span_days=float(df['jd'].max() - df['jd'].min()))
    if 'mag' in df.columns and df['mag'].notna().sum() >= 3:
        mag = df['mag'].to_numpy(dtype=float)
        out.update(lc_median_mag=float(np.nanmedian(mag)), lc_amp_mag_p95_p05=float(np.nanpercentile(mag, 95) - np.nanpercentile(mag, 5)), lc_mag_slope_per_year=robust_slope_per_year(df['jd'].to_numpy(dtype=float), mag))
    if 'flux' in df.columns and df['flux'].notna().sum() >= 3:
        flux = df['flux'].to_numpy(dtype=float)
        out.update(lc_flux_median=float(np.nanmedian(flux)), lc_flux_amp_p95_p05=float(np.nanpercentile(flux, 95) - np.nanpercentile(flux, 5)), lc_flux_slope_per_year=robust_slope_per_year(df['jd'].to_numpy(dtype=float), flux))
        try:
            sigma = df['flux_err'].to_numpy(dtype=float) if 'flux_err' in df.columns else None
            edges = bayesian_blocks(df['jd'].to_numpy(dtype=float), flux, sigma=sigma, fitness='measures')
            out['lc_bb_n_blocks'] = int(max(0, len(edges) - 1))
            block_medians = [float(np.nanmedian(df.loc[(df['jd'] >= lo) & (df['jd'] < hi), 'flux'])) for lo, hi in zip(edges[:-1], edges[1:]) if ((df['jd'] >= lo) & (df['jd'] < hi)).any()]
            if len(block_medians) >= 2:
                out['lc_bb_largest_jump_flux'] = float(np.nanmax(np.abs(np.diff(block_medians))))
        except Exception:
            pass
    return out

def process_lightcurve_table(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in frame.iterrows():
        result = row.to_dict()
        path = str(row.get('lc_path') or '').strip()
        if not path or path.lower() == 'nan' or not Path(path).exists():
            result.update(summarize_clean_lightcurve(pd.DataFrame()))
            result.update(lc_process_status='missing_lc_path', lc_process_error='')
            rows.append(result)
            continue
        try:
            clean = clean_asassn_lightcurve(read_skypatrol_lightcurve(path))
            result.update(summarize_clean_lightcurve(clean))
            result['clean_lc_path'] = write_cleaned_lightcurve(row, clean)
            result.update(lc_process_status='ok', lc_process_error='')
        except Exception as exc:
            result.update(summarize_clean_lightcurve(pd.DataFrame()))
            result.update(lc_process_status='process_failed', lc_process_error=f'{type(exc).__name__}: {exc}')
        rows.append(result)
    return pd.DataFrame(rows)


In [10]:
resolved_path = PROCESSED_DIR / 'clagn_asassn_resolved.csv'
fetched_path = PROCESSED_DIR / 'clagn_asassn_fetched.csv'
processed_path = PROCESSED_DIR / 'clagn_asassn_lc_metrics.csv'

if RUN_ASASSN_RESOLUTION:
    if clagn_sensitive.empty:
        raise RuntimeError('No ASAS-SN-sensitive CLAGN table is loaded.')
    clagn_resolved = resolve_catalog_sources(clagn_sensitive, max_objects=MAX_OBJECTS)
    clagn_resolved.to_csv(resolved_path, index=False)
elif resolved_path.exists():
    clagn_resolved = pd.read_csv(resolved_path)
else:
    clagn_resolved = pd.DataFrame()

if RUN_ASASSN_FETCH:
    if clagn_resolved.empty:
        raise RuntimeError('No resolved ASAS-SN source table is loaded.')
    clagn_fetched = fetch_resolved_lightcurves(clagn_resolved)
    clagn_fetched.to_csv(fetched_path, index=False)
elif fetched_path.exists():
    clagn_fetched = pd.read_csv(fetched_path)
else:
    clagn_fetched = clagn_resolved.copy() if not clagn_resolved.empty else pd.DataFrame()

if RUN_LIGHTCURVE_PROCESSING:
    if clagn_fetched.empty:
        raise RuntimeError('No fetched light-curve table is loaded.')
    clagn_lc_metrics = process_lightcurve_table(clagn_fetched)
    clagn_lc_metrics.to_csv(processed_path, index=False)
elif processed_path.exists():
    clagn_lc_metrics = pd.read_csv(processed_path)
else:
    clagn_lc_metrics = pd.DataFrame()

display(Markdown(f'**Resolved:** {len(clagn_resolved):,}; **Fetched rows:** {len(clagn_fetched):,}; **Processed metrics:** {len(clagn_lc_metrics):,}'))
clagn_lc_metrics.head(10) if not clagn_lc_metrics.empty else clagn_fetched.head(10)


**Resolved:** 0; **Fetched rows:** 0; **Processed metrics:** 0

""


## Precursor Science Metrics And Review Queue

This stage adds transition metadata, computes precursor-focused light-curve metrics from cleaned ASAS-SN tables, and ranks objects for visual review. It never infers spectral transition dates from the photometry itself.

In [11]:
TRANSITION_COLUMNS = ['transition_direction','transition_mjd_min','transition_mjd_max','transition_mjd_anchor','transition_date_source','transition_date_quality']

# Add curated literature windows here as they are verified. Keys may be clagn_id or any alias/name token.
MANUAL_TRANSITION_WINDOWS: dict[str, dict] = {}

def pipe_values(value) -> list[str]:
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    return [part.strip() for part in str(value).split('|') if part.strip()]

def infer_transition_direction(row: pd.Series) -> str:
    text = ' '.join(pipe_values(row.get('clagn_types', '')) + pipe_values(row.get('source_labels', '')) + pipe_values(row.get('source_keys', '')) + [str(row.get('name', '')), str(row.get('aliases', ''))]).lower()
    has_on = bool(re.search(r'turn[_ -]?on|type[_ -]?1|bright', text))
    has_off = bool(re.search(r'turn[_ -]?off|type[_ -]?2|dim', text))
    if has_on and has_off:
        return 'mixed'
    if has_on:
        return 'turn_on'
    if has_off:
        return 'turn_off'
    return 'unknown'

def _finite_series_values(row: pd.Series, columns: tuple[str, ...]) -> list[float]:
    values = []
    for col in columns:
        if col in row.index:
            value = finite_float(row.get(col))
            if math.isfinite(value):
                values.append(value)
    return values

def extract_transition_window_from_row(row: pd.Series) -> dict:
    direct_min = _finite_series_values(row, ('transition_mjd_min', 'mjd_min', 'transition_start_mjd', 'spec_mjd_1', 'first_spec_mjd'))
    direct_max = _finite_series_values(row, ('transition_mjd_max', 'mjd_max', 'transition_end_mjd', 'spec_mjd_2', 'second_spec_mjd'))
    anchors = _finite_series_values(row, ('transition_mjd_anchor', 'transition_mjd', 'event_mjd'))
    values = []
    if direct_min:
        values.append(min(direct_min))
    if direct_max:
        values.append(max(direct_max))
    if not values and anchors:
        values = anchors[:1]
    if not values:
        return {}
    mjd_min = min(values)
    mjd_max = max(values)
    anchor = float(np.nanmean([mjd_min, mjd_max])) if math.isfinite(mjd_min) and math.isfinite(mjd_max) else (anchors[0] if anchors else math.nan)
    return dict(transition_mjd_min=mjd_min, transition_mjd_max=mjd_max, transition_mjd_anchor=anchor, transition_date_source='catalog_columns', transition_date_quality='inferred_ref')

def manual_transition_window_for_row(row: pd.Series) -> dict:
    tokens = [str(row.get('clagn_id', '')).strip(), str(row.get('name', '')).strip()]
    tokens.extend(pipe_values(row.get('aliases', '')))
    for token in tokens:
        if token in MANUAL_TRANSITION_WINDOWS:
            return MANUAL_TRANSITION_WINDOWS[token].copy()
    return {}

def build_transition_metadata(frame: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for _, row in frame.iterrows():
        metadata = {col: math.nan for col in TRANSITION_COLUMNS}
        metadata.update(transition_direction=infer_transition_direction(row), transition_date_source='', transition_date_quality='missing')
        inferred = extract_transition_window_from_row(row)
        manual = manual_transition_window_for_row(row)
        if inferred:
            metadata.update(inferred)
        if manual:
            metadata.update(manual)
            metadata.setdefault('transition_date_quality', 'inferred_ref')
            metadata.setdefault('transition_date_source', 'manual_literature_window')
        mjd_min = finite_float(metadata.get('transition_mjd_min'))
        mjd_max = finite_float(metadata.get('transition_mjd_max'))
        anchor = finite_float(metadata.get('transition_mjd_anchor'))
        if not math.isfinite(anchor) and math.isfinite(mjd_min) and math.isfinite(mjd_max):
            metadata['transition_mjd_anchor'] = float((mjd_min + mjd_max) / 2)
        elif not math.isfinite(anchor) and math.isfinite(mjd_min):
            metadata['transition_mjd_anchor'] = mjd_min
        if math.isfinite(finite_float(metadata.get('transition_mjd_anchor'))):
            if not metadata.get('transition_date_source'):
                metadata['transition_date_source'] = 'catalog_or_manual_reference'
            if metadata.get('transition_date_quality') == 'missing':
                metadata['transition_date_quality'] = 'inferred_ref'
        metadata.update(clagn_id=row.get('clagn_id', ''), name=row.get('name', ''), aliases=row.get('aliases', ''), source_keys=row.get('source_keys', ''), clagn_types=row.get('clagn_types', ''))
        rows.append(metadata)
    columns = ['clagn_id','name','aliases','source_keys','clagn_types'] + TRANSITION_COLUMNS
    return pd.DataFrame(rows).reindex(columns=columns)

def attach_transition_metadata(frame: pd.DataFrame, transition_metadata: pd.DataFrame) -> pd.DataFrame:
    if frame.empty:
        return frame.copy()
    out = frame.drop(columns=[col for col in TRANSITION_COLUMNS if col in frame.columns], errors='ignore').copy()
    if transition_metadata.empty or 'clagn_id' not in out.columns:
        for col in TRANSITION_COLUMNS:
            if col not in out.columns:
                out[col] = math.nan if col.startswith('transition_mjd') else ('missing' if col == 'transition_date_quality' else '')
        return out
    keep = ['clagn_id'] + [col for col in TRANSITION_COLUMNS if col in transition_metadata.columns]
    return out.merge(transition_metadata[keep], on='clagn_id', how='left')

def robust_mad(values) -> float:
    arr = np.asarray(values, dtype=float)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return math.nan
    med = np.nanmedian(arr)
    return float(1.4826 * np.nanmedian(np.abs(arr - med)))

def weighted_slope_per_year(x_jd, y, yerr=None) -> tuple[float, float]:
    x = np.asarray(x_jd, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    if yerr is not None:
        err = np.asarray(yerr, dtype=float)
        mask &= np.isfinite(err) & (err > 0)
    else:
        err = None
    if mask.sum() < 3 or np.nanmax(x[mask]) == np.nanmin(x[mask]):
        return math.nan, math.nan
    x_year = (x[mask] - np.nanmean(x[mask])) / 365.25
    design = np.vstack([x_year, np.ones(mask.sum())]).T
    try:
        if err is None:
            beta, residuals, _, _ = np.linalg.lstsq(design, y[mask], rcond=None)
            dof = max(1, mask.sum() - 2)
            scatter2 = float(residuals[0] / dof) if len(residuals) else float(np.nanvar(y[mask] - design @ beta))
            cov = scatter2 * np.linalg.inv(design.T @ design)
        else:
            weights = 1.0 / np.square(err[mask])
            normal = design.T @ (weights[:, None] * design)
            cov = np.linalg.inv(normal)
            beta = cov @ (design.T @ (weights * y[mask]))
        slope = float(beta[0])
        slope_err = float(math.sqrt(max(cov[0, 0], 0)))
        return slope, float(slope / slope_err) if slope_err > 0 else math.nan
    except Exception:
        return math.nan, math.nan

def reduced_chi_square(values, errors) -> float:
    y = np.asarray(values, dtype=float)
    err = np.asarray(errors, dtype=float)
    mask = np.isfinite(y) & np.isfinite(err) & (err > 0)
    if mask.sum() < 3:
        return math.nan
    median = np.nanmedian(y[mask])
    return float(np.nansum(np.square((y[mask] - median) / err[mask])) / max(1, mask.sum() - 1))

def normalized_excess_variance(values, errors) -> float:
    y = np.asarray(values, dtype=float)
    err = np.asarray(errors, dtype=float)
    mask = np.isfinite(y) & np.isfinite(err) & (err > 0)
    if mask.sum() < 3:
        return math.nan
    mean_flux = float(np.nanmean(y[mask]))
    if mean_flux == 0 or not math.isfinite(mean_flux):
        return math.nan
    sample_var = float(np.nanvar(y[mask], ddof=1))
    noise_var = float(np.nanmean(np.square(err[mask])))
    return float((sample_var - noise_var) / (mean_flux * mean_flux))

def rolling_anomaly_metrics(df: pd.DataFrame, *, window: int = ROLLING_ANOMALY_WINDOW, sigma: float = ROLLING_ANOMALY_SIGMA) -> dict:
    total, count = 0, 0
    if df.empty or 'flux' not in df.columns:
        return dict(science_rolling_anomaly_count=0, science_rolling_anomaly_fraction=math.nan)
    for _, band_df in df.sort_values('jd').groupby('band'):
        y = band_df['flux'].astype(float).reset_index(drop=True)
        if y.notna().sum() < max(5, window // 3):
            continue
        min_periods = max(5, min(window // 3, len(y)))
        rolling_median = y.rolling(window, center=True, min_periods=min_periods).median()
        rolling_scatter = y.rolling(window, center=True, min_periods=min_periods).apply(robust_mad, raw=True)
        fallback_scatter = robust_mad(y)
        scatter = rolling_scatter.replace(0, np.nan).fillna(fallback_scatter)
        z = np.abs((y - rolling_median) / scatter)
        finite = np.isfinite(z)
        total += int(finite.sum())
        count += int((z[finite] >= sigma).sum())
    return dict(science_rolling_anomaly_count=count, science_rolling_anomaly_fraction=float(count / total) if total else math.nan)

def season_summary_table(df: pd.DataFrame, *, min_points: int = MIN_SEASON_POINTS) -> pd.DataFrame:
    if df.empty or 'flux' not in df.columns:
        return pd.DataFrame()
    work = df[np.isfinite(df['jd']) & np.isfinite(df['flux'])].copy()
    if work.empty:
        return pd.DataFrame()
    work['season_year'] = np.floor((work['jd'] - 2451545.0) / 365.25 + 2000).astype(int)
    rows = []
    for season_year, group in work.groupby('season_year'):
        if len(group) < min_points:
            continue
        rows.append(dict(season_year=int(season_year), season_jd=float(np.nanmedian(group['jd'])), season_n=int(len(group)), season_flux_median=float(np.nanmedian(group['flux'])), season_flux_mad=robust_mad(group['flux']), season_mag_median=float(np.nanmedian(group['mag'])) if 'mag' in group.columns and group['mag'].notna().any() else math.nan))
    return pd.DataFrame(rows).sort_values('season_jd').reset_index(drop=True) if rows else pd.DataFrame()

def season_change_metrics(df: pd.DataFrame) -> dict:
    seasons = season_summary_table(df)
    out = dict(science_season_count=len(seasons), science_max_season_delta_flux=math.nan, science_max_season_delta_frac=math.nan, science_season_slope_flux_per_year=math.nan, science_season_slope_snr=math.nan)
    if seasons.empty:
        return out
    medians = seasons['season_flux_median'].to_numpy(dtype=float)
    if len(medians) >= 2:
        deltas = np.diff(medians)
        largest = float(deltas[np.nanargmax(np.abs(deltas))])
        scale = abs(float(np.nanmedian(medians))) or robust_mad(medians) or 1.0
        out.update(science_max_season_delta_flux=largest, science_max_season_delta_frac=float(largest / scale))
    slope, snr = weighted_slope_per_year(seasons['season_jd'].to_numpy(dtype=float), medians)
    out.update(science_season_slope_flux_per_year=slope, science_season_slope_snr=snr)
    return out

def bayesian_block_metrics(df: pd.DataFrame) -> dict:
    out = dict(science_bb_n_blocks=math.nan, science_bb_largest_jump_flux=math.nan, science_bb_largest_jump_frac=math.nan)
    if df.empty or 'flux' not in df.columns:
        return out
    work = df[np.isfinite(df['jd']) & np.isfinite(df['flux'])].sort_values('jd')
    if len(work) < 5:
        return out
    try:
        sigma = work['flux_err'].to_numpy(dtype=float) if 'flux_err' in work.columns else None
        edges = bayesian_blocks(work['jd'].to_numpy(dtype=float), work['flux'].to_numpy(dtype=float), sigma=sigma, fitness='measures')
        block_medians = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            mask = (work['jd'] >= lo) & (work['jd'] < hi)
            if mask.any():
                block_medians.append(float(np.nanmedian(work.loc[mask, 'flux'])))
        out['science_bb_n_blocks'] = int(max(0, len(edges) - 1))
        if len(block_medians) >= 2:
            largest = float(np.nanmax(np.abs(np.diff(block_medians))))
            scale = abs(float(np.nanmedian(block_medians))) or robust_mad(block_medians) or 1.0
            out.update(science_bb_largest_jump_flux=largest, science_bb_largest_jump_frac=float(largest / scale))
    except Exception:
        pass
    return out

def empty_transition_window_metrics() -> dict:
    out = {}
    for years in PRECURSOR_WINDOWS_YEARS:
        prefix = f'pre_{years}yr'
        out.update({f'{prefix}_n_points': 0, f'{prefix}_median_flux': math.nan, f'{prefix}_flux_slope_per_year': math.nan, f'{prefix}_flux_slope_snr': math.nan, f'{prefix}_baseline_delta_flux': math.nan, f'{prefix}_baseline_delta_frac': math.nan, f'{prefix}_rolling_anomaly_count': 0})
    return out

def compute_transition_window_metrics(df: pd.DataFrame, row: pd.Series) -> dict:
    out = empty_transition_window_metrics()
    mjd_min = finite_float(row.get('transition_mjd_min'))
    if not math.isfinite(mjd_min) or df.empty or 'flux' not in df.columns:
        return out
    transition_jd = mjd_min + 2400000.5
    for years in PRECURSOR_WINDOWS_YEARS:
        prefix = f'pre_{years}yr'
        start_jd = transition_jd - years * 365.25
        window = df[(df['jd'] >= start_jd) & (df['jd'] < transition_jd)].copy()
        baseline = df[df['jd'] < start_jd].copy()
        out[f'{prefix}_n_points'] = int(len(window))
        if len(window) >= 3 and window['flux'].notna().sum() >= 3:
            median_flux = float(np.nanmedian(window['flux']))
            slope, snr = weighted_slope_per_year(window['jd'], window['flux'], window['flux_err'] if 'flux_err' in window.columns else None)
            out[f'{prefix}_median_flux'] = median_flux
            out[f'{prefix}_flux_slope_per_year'] = slope
            out[f'{prefix}_flux_slope_snr'] = snr
            out[f'{prefix}_rolling_anomaly_count'] = rolling_anomaly_metrics(window)['science_rolling_anomaly_count']
            if len(baseline) >= 3 and baseline['flux'].notna().sum() >= 3:
                baseline_median = float(np.nanmedian(baseline['flux']))
                delta = median_flux - baseline_median
                scale = abs(baseline_median) or robust_mad(baseline['flux']) or 1.0
                out[f'{prefix}_baseline_delta_flux'] = float(delta)
                out[f'{prefix}_baseline_delta_frac'] = float(delta / scale)
    return out

def summarize_precursor_lightcurve(df: pd.DataFrame) -> dict:
    out = dict(science_n_points=len(df), science_n_bands=0, science_jd_start=math.nan, science_jd_end=math.nan, science_coverage_days=math.nan, science_coverage_years=math.nan, science_median_cadence_days=math.nan, science_flux_median=math.nan, science_flux_mad=math.nan, science_flux_amp_p95_p05=math.nan, science_flux_frac_amp_p95_p05=math.nan, science_flux_slope_per_year=math.nan, science_flux_slope_snr=math.nan, science_mag_median=math.nan, science_mag_amp_p95_p05=math.nan, science_reduced_chi2=math.nan, science_excess_variance=math.nan)
    out.update(rolling_anomaly_metrics(df))
    out.update(season_change_metrics(df))
    out.update(bayesian_block_metrics(df))
    if df.empty:
        return out
    out.update(science_n_bands=int(df['band'].nunique()) if 'band' in df.columns else 0, science_jd_start=float(df['jd'].min()), science_jd_end=float(df['jd'].max()), science_coverage_days=float(df['jd'].max() - df['jd'].min()), science_coverage_years=float((df['jd'].max() - df['jd'].min()) / 365.25))
    if len(df) >= 2:
        out['science_median_cadence_days'] = float(np.nanmedian(np.diff(np.sort(df['jd'].to_numpy(dtype=float)))))
    if 'flux' in df.columns and df['flux'].notna().sum() >= 3:
        flux = df['flux'].to_numpy(dtype=float)
        flux_median = float(np.nanmedian(flux))
        flux_amp = float(np.nanpercentile(flux, 95) - np.nanpercentile(flux, 5))
        flux_scale = abs(flux_median) or robust_mad(flux) or 1.0
        slope, snr = weighted_slope_per_year(df['jd'], df['flux'], df['flux_err'] if 'flux_err' in df.columns else None)
        out.update(science_flux_median=flux_median, science_flux_mad=robust_mad(flux), science_flux_amp_p95_p05=flux_amp, science_flux_frac_amp_p95_p05=float(flux_amp / flux_scale), science_flux_slope_per_year=slope, science_flux_slope_snr=snr)
        if 'flux_err' in df.columns:
            out.update(science_reduced_chi2=reduced_chi_square(df['flux'], df['flux_err']), science_excess_variance=normalized_excess_variance(df['flux'], df['flux_err']))
    if 'mag' in df.columns and df['mag'].notna().sum() >= 3:
        mag = df['mag'].to_numpy(dtype=float)
        out.update(science_mag_median=float(np.nanmedian(mag)), science_mag_amp_p95_p05=float(np.nanpercentile(mag, 95) - np.nanpercentile(mag, 5)))
    return out

def load_cleaned_lightcurve_for_row(row: pd.Series) -> pd.DataFrame:
    clean_path = str(row.get('clean_lc_path') or '').strip()
    if clean_path and clean_path.lower() != 'nan' and Path(clean_path).exists():
        return clean_asassn_lightcurve(read_skypatrol_lightcurve(clean_path))
    raw_path = str(row.get('lc_path') or '').strip()
    if raw_path and raw_path.lower() != 'nan' and Path(raw_path).exists():
        return clean_asassn_lightcurve(read_skypatrol_lightcurve(raw_path))
    return pd.DataFrame()

def compute_precursor_science_metrics(frame: pd.DataFrame, transition_metadata: pd.DataFrame | None = None) -> pd.DataFrame:
    if frame.empty:
        return pd.DataFrame()
    work = attach_transition_metadata(frame, transition_metadata) if transition_metadata is not None else frame.copy()
    rows = []
    for _, row in work.iterrows():
        result = row.to_dict()
        try:
            clean = load_cleaned_lightcurve_for_row(row)
            if clean.empty:
                result.update(summarize_precursor_lightcurve(clean))
                result.update(empty_transition_window_metrics())
                result.update(science_status='missing_or_empty_lc', science_error='')
            else:
                result.update(summarize_precursor_lightcurve(clean))
                result.update(compute_transition_window_metrics(clean, row))
                result.update(science_status='ok', science_error='')
        except Exception as exc:
            result.update(summarize_precursor_lightcurve(pd.DataFrame()))
            result.update(empty_transition_window_metrics())
            result.update(science_status='science_failed', science_error=f'{type(exc).__name__}: {exc}')
        rows.append(result)
    return pd.DataFrame(rows)

def percentile_component(series: pd.Series) -> pd.Series:
    values = pd.to_numeric(series, errors='coerce').replace([np.inf, -np.inf], np.nan)
    out = pd.Series(0.0, index=series.index, dtype=float)
    mask = values.notna()
    if mask.sum() == 0:
        return out
    if mask.sum() == 1:
        out.loc[mask] = 1.0
    else:
        out.loc[mask] = values.loc[mask].rank(pct=True).astype(float)
    return out.fillna(0.0)

def rank_precursor_candidates(metrics: pd.DataFrame) -> pd.DataFrame:
    if metrics.empty:
        return pd.DataFrame()
    out = metrics.copy()
    amp_component = percentile_component(out.get('science_flux_frac_amp_p95_p05', pd.Series(index=out.index, dtype=float)))
    trend_component = percentile_component(pd.to_numeric(out.get('science_flux_slope_snr', pd.Series(index=out.index, dtype=float)), errors='coerce').abs())
    excess_component = percentile_component(out.get('science_excess_variance', pd.Series(index=out.index, dtype=float)))
    block_component = percentile_component(out.get('science_bb_largest_jump_frac', pd.Series(index=out.index, dtype=float)))
    anomaly_component = percentile_component(out.get('science_rolling_anomaly_fraction', pd.Series(index=out.index, dtype=float)))
    coverage_input = np.log1p(pd.to_numeric(out.get('science_n_points', pd.Series(index=out.index, dtype=float)), errors='coerce').fillna(0)) + pd.to_numeric(out.get('science_coverage_years', pd.Series(index=out.index, dtype=float)), errors='coerce').fillna(0)
    coverage_component = percentile_component(coverage_input)
    out['rank_component_amplitude'] = amp_component
    out['rank_component_trend'] = trend_component
    out['rank_component_excess_variance'] = excess_component
    out['rank_component_bayesian_blocks'] = block_component
    out['rank_component_anomaly'] = anomaly_component
    out['rank_component_coverage'] = coverage_component
    score = 0.22 * amp_component + 0.20 * trend_component + 0.18 * excess_component + 0.18 * block_component + 0.12 * anomaly_component + 0.10 * coverage_component
    good_status = out.get('science_status', '').astype(str).eq('ok') if 'science_status' in out.columns else pd.Series(True, index=out.index)
    out['precursor_score'] = score.where(good_status, 0.0).clip(0, 1)
    out['review_priority'] = np.select([out['precursor_score'] >= 0.75, out['precursor_score'] >= 0.45], ['high', 'medium'], default='low')
    sort_cols = ['precursor_score', 'science_flux_frac_amp_p95_p05', 'science_coverage_years']
    sort_cols = [col for col in sort_cols if col in out.columns]
    return out.sort_values(sort_cols, ascending=[False] * len(sort_cols)).reset_index(drop=True)

def plot_precursor_metric_distributions(review_queue: pd.DataFrame) -> go.Figure:
    fig = make_subplots(rows=1, cols=2, subplot_titles=('Review score', 'Amplitude vs trend'))
    if review_queue.empty:
        fig.update_layout(height=420, title='CLAGN precursor review metrics')
        return fig
    fig.add_trace(go.Histogram(x=review_queue['precursor_score'], nbinsx=20, name='score'), row=1, col=1)
    fig.add_trace(go.Scatter(x=review_queue.get('science_flux_frac_amp_p95_p05'), y=pd.to_numeric(review_queue.get('science_flux_slope_snr'), errors='coerce').abs(), mode='markers', text=review_queue.get('clagn_id'), marker=dict(color=review_queue['precursor_score'], colorscale='Viridis', showscale=True, colorbar=dict(title='score')), name='objects'), row=1, col=2)
    fig.update_xaxes(title_text='precursor score', row=1, col=1)
    fig.update_xaxes(title_text='flux p95-p05 / median', row=1, col=2)
    fig.update_yaxes(title_text='abs flux slope S/N', row=1, col=2)
    fig.update_layout(height=420, title='CLAGN precursor review metrics', showlegend=False)
    return fig

def plot_clagn_precursor_diagnostic(row: pd.Series, *, cleaned: bool = True) -> go.Figure:
    df = load_cleaned_lightcurve_for_row(row) if cleaned else read_skypatrol_lightcurve(row.get('lc_path'))
    if df.empty:
        raise FileNotFoundError(f'No usable light curve for {row.get("clagn_id", "row")}')
    fig = go.Figure()
    for band, band_df in df.groupby('band'):
        fig.add_trace(go.Scattergl(x=band_df['jd'], y=band_df['flux'], error_y=dict(type='data', array=band_df['flux_err']) if 'flux_err' in band_df.columns else None, mode='markers', name=f'{band} points', marker=dict(size=5, opacity=0.65)))
    seasons = season_summary_table(df)
    if not seasons.empty:
        fig.add_trace(go.Scatter(x=seasons['season_jd'], y=seasons['season_flux_median'], mode='lines+markers', name='season median', line=dict(width=3, color='black'), marker=dict(size=8)))
    try:
        work = df[np.isfinite(df['jd']) & np.isfinite(df['flux'])].sort_values('jd')
        if len(work) >= 5:
            sigma = work['flux_err'].to_numpy(dtype=float) if 'flux_err' in work.columns else None
            edges = bayesian_blocks(work['jd'].to_numpy(dtype=float), work['flux'].to_numpy(dtype=float), sigma=sigma, fitness='measures')
            for edge in edges[1:-1]:
                fig.add_vline(x=float(edge), line_width=1, line_dash='dot', line_color='gray')
    except Exception:
        pass
    mjd_min = finite_float(row.get('transition_mjd_min'))
    mjd_max = finite_float(row.get('transition_mjd_max'))
    anchor = finite_float(row.get('transition_mjd_anchor'))
    if math.isfinite(mjd_min) and math.isfinite(mjd_max):
        fig.add_vrect(x0=mjd_min + 2400000.5, x1=mjd_max + 2400000.5, fillcolor='LightSalmon', opacity=0.25, line_width=0, annotation_text='transition window', annotation_position='top left')
    elif math.isfinite(anchor):
        fig.add_vline(x=anchor + 2400000.5, line_width=2, line_dash='dash', line_color='firebrick')
    title = f"{row.get('clagn_id', '')} {row.get('name', '')}".strip()
    score = finite_float(row.get('precursor_score'))
    if math.isfinite(score):
        title = f'{title} | score={score:.2f}'
    fig.update_layout(title=title, xaxis_title='JD', yaxis_title='ASAS-SN flux', height=520)
    return fig


In [12]:
transition_metadata_path = PROCESSED_DIR / 'clagn_transition_metadata.csv'
precursor_metrics_path = PROCESSED_DIR / 'clagn_precursor_science_metrics.csv'
review_queue_path = PROCESSED_DIR / 'clagn_precursor_review_queue.csv'

science_input = clagn_lc_metrics.copy() if not clagn_lc_metrics.empty else (clagn_fetched.copy() if not clagn_fetched.empty else pd.DataFrame())
transition_input = science_input.copy() if not science_input.empty else (clagn_sensitive.copy() if not clagn_sensitive.empty else pd.DataFrame())

if RUN_TRANSITION_METADATA:
    transition_metadata = build_transition_metadata(transition_input)
    transition_metadata.to_csv(transition_metadata_path, index=False)
elif transition_metadata_path.exists():
    transition_metadata = pd.read_csv(transition_metadata_path)
elif not transition_input.empty:
    transition_metadata = build_transition_metadata(transition_input)
else:
    transition_metadata = pd.DataFrame(columns=['clagn_id','name','aliases','source_keys','clagn_types'] + TRANSITION_COLUMNS)

if RUN_PRECURSOR_SCIENCE:
    if science_input.empty:
        raise RuntimeError('No processed or fetched light-curve table is loaded.')
    if transition_metadata.empty:
        transition_metadata = build_transition_metadata(science_input)
    clagn_precursor_metrics = compute_precursor_science_metrics(science_input, transition_metadata)
    clagn_precursor_metrics.to_csv(precursor_metrics_path, index=False)
    clagn_precursor_review_queue = rank_precursor_candidates(clagn_precursor_metrics)
    clagn_precursor_review_queue.to_csv(review_queue_path, index=False)
elif precursor_metrics_path.exists():
    clagn_precursor_metrics = pd.read_csv(precursor_metrics_path)
    clagn_precursor_review_queue = pd.read_csv(review_queue_path) if review_queue_path.exists() else rank_precursor_candidates(clagn_precursor_metrics)
else:
    clagn_precursor_metrics = pd.DataFrame()
    clagn_precursor_review_queue = pd.DataFrame()

n_transition_dates = int(pd.to_numeric(transition_metadata.get('transition_mjd_anchor', pd.Series(dtype=float)), errors='coerce').notna().sum()) if not transition_metadata.empty else 0
n_science_ok = int(clagn_precursor_metrics.get('science_status', pd.Series(dtype=str)).astype(str).eq('ok').sum()) if not clagn_precursor_metrics.empty else 0
display(Markdown(f'**Transition metadata rows:** {len(transition_metadata):,} ({n_transition_dates:,} with date anchors); **science rows:** {len(clagn_precursor_metrics):,}; **science ok:** {n_science_ok:,}'))

if not clagn_precursor_review_queue.empty:
    display_cols = ['clagn_id','name','precursor_score','review_priority','transition_direction','transition_date_quality','science_n_points','science_coverage_years','science_flux_frac_amp_p95_p05','science_flux_slope_snr','science_excess_variance','science_bb_largest_jump_frac','science_rolling_anomaly_fraction']
    display(clagn_precursor_review_queue[[col for col in display_cols if col in clagn_precursor_review_queue.columns]].head(REVIEW_TOP_N))
    if RUN_REVIEW_PLOTS:
        display(plot_precursor_metric_distributions(clagn_precursor_review_queue))
elif not transition_metadata.empty:
    display(transition_metadata.head(10))


**Transition metadata rows:** 0 (0 with date anchors); **science rows:** 0; **science ok:** 0

In [13]:
def run_precursor_metric_self_check() -> pd.DataFrame:
    rng = np.random.default_rng(7)
    jd = np.linspace(2457000.0, 2457000.0 + 7 * 365.25, 260)
    trend = 0.7 * ((jd - jd.min()) / 365.25)
    step = np.where(jd > jd.min() + 4.0 * 365.25, 5.0, 0.0)
    flux = 10.0 + trend + step + rng.normal(0, 0.15, len(jd))
    flux[145] += 15.0
    synthetic = pd.DataFrame(dict(jd=jd, flux=flux, flux_err=np.full(len(jd), 0.2), mag=18.0 - 2.5 * np.log10(np.maximum(flux, 0.01) / 10.0), mag_err=np.full(len(jd), 0.03), band='g', quality='G'))
    summary = summarize_precursor_lightcurve(synthetic)
    transition_row = pd.Series({'transition_mjd_min': jd.min() + 5.5 * 365.25 - 2400000.5, 'transition_mjd_max': jd.min() + 5.6 * 365.25 - 2400000.5, 'transition_mjd_anchor': jd.min() + 5.55 * 365.25 - 2400000.5})
    window_metrics = compute_transition_window_metrics(synthetic, transition_row)
    ranked = rank_precursor_candidates(pd.DataFrame([{**summary, **window_metrics, 'clagn_id': 'SYNTHETIC', 'name': 'synthetic step trend anomaly', 'science_status': 'ok'}]))
    checks = dict(
        slope_finite=math.isfinite(summary['science_flux_slope_per_year']),
        step_metric_finite=math.isfinite(summary['science_max_season_delta_flux']),
        anomaly_detected=summary['science_rolling_anomaly_count'] > 0,
        transition_window_finite=math.isfinite(window_metrics['pre_5yr_median_flux']),
        rank_score_finite=math.isfinite(float(ranked.loc[0, 'precursor_score'])),
    )
    if not all(checks.values()):
        raise AssertionError(f'Precursor science self-check failed: {checks}')
    return pd.DataFrame([{**checks, 'synthetic_score': float(ranked.loc[0, 'precursor_score']), 'synthetic_slope': summary['science_flux_slope_per_year'], 'synthetic_anomaly_count': summary['science_rolling_anomaly_count']}])

precursor_self_check = run_precursor_metric_self_check()
precursor_self_check


,slope_finite,step_metric_finite,anomaly_detected,transition_window_finite,rank_score_finite,synthetic_score,synthetic_slope,synthetic_anomaly_count
0,True,True,True,True,True,1.0,1.750774,1


In [14]:
def plot_asassn_lightcurve_for_row(row: pd.Series, *, cleaned: bool = True) -> go.Figure:
    path = str(row.get('lc_path') or '').strip()
    if not path or not Path(path).exists():
        raise FileNotFoundError(f'No local light curve path for {row.get("clagn_id", "row")}')
    df = read_skypatrol_lightcurve(path)
    if cleaned:
        df = clean_asassn_lightcurve(df)
    fig = go.Figure()
    for band, band_df in df.groupby('band'):
        fig.add_trace(go.Scattergl(x=band_df['jd'], y=band_df['mag'], mode='markers', name=str(band), marker=dict(size=5)))
    fig.update_layout(title=f"{row.get('clagn_id', '')} {row.get('name', '')}".strip(), xaxis_title='JD', yaxis_title='ASAS-SN mag', yaxis_autorange='reversed', height=480)
    return fig

# Example after fetching:
# plot_asassn_lightcurve_for_row(clagn_fetched.iloc[0])
